## PTI and Washing

### E-form Dataset

In [1]:
from datetime import datetime
import polars as pl

from data_source.sheet_ids import (
    LOGISTICS_SHEET_ID,
    CONTAINER_CLEANING_SHEET,
    CONTAINER_PTI_SHEET,
    CONTAINER_GATE_IN_SHEET
)
from data_source.make_dataset import load_gsheet_data
from type_casting.containers import containers_enum

from dataframe.emr import washing,pti

In [2]:
from dataframe.stuffing import coa

## Container Gate In data

In [2]:
gate_in_raw = load_gsheet_data(LOGISTICS_SHEET_ID,CONTAINER_GATE_IN_SHEET)

gate_in_df = gate_in_raw.filter(pl.col("Date").dt.date().ne(datetime.today().date())).select(
    pl.col("Date").alias("date"),
    pl.col("Time").alias("time"),
    pl.col("Container Number").cast(containers_enum).alias("container_number"),
    pl.col("Shipping Line").cast(pl.Categorical).alias("shipping_line"),
    pl.col("Type").cast(pl.Categorical).alias("type"),
    pl.col("Unit Manufacturer").cast(pl.Categorical).alias("unit_manufacturer"),
    pl.col("Status").alias("status"),
    pl.col("PTI Status").alias("pti_status"),
)


In [4]:
non_pti_gate_in = gate_in_df.filter(pl.col("pti_status").eq("NON PTI"))

current_month_non_pti = non_pti_gate_in.filter(pl.col("date").dt.year().eq(2025).and_(pl.col("date").dt.month().eq(6))).collect()

In [5]:
gate_in_df.filter(pl.col("container_number").eq(pl.lit("SUDU8191258"))).collect()

date,time,container_number,shipping_line,type,unit_manufacturer,status,pti_status
date,time,enum,cat,cat,cat,str,str


### Load the datasets lazyly

In [6]:

import requests
from io import StringIO


def load_raw_sheet(sheet_id: str, sheet_name: str) -> pl.LazyFrame:
    """
    Loads a Google Sheet as a Polars LazyFrame.

    Args:
        sheet_id (str): The ID of the Google Sheet.
        sheet_name (str): The name of the sheet to load.

    Returns:
        pl.LazyFrame: A LazyFrame containing the sheet data, or None if an error occurred.
    """
    link:str = "https://docs.google.com/spreadsheets"
    url:str = f"{link}/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        # Use StringIO to avoid creating a temporary file
        csv_data = StringIO(response.text)
        return pl.read_csv(csv_data, try_parse_dates=False).lazy()

    except requests.exceptions.RequestException as e:
        print(
            "An error occurred while trying to access the Google Sheet: %s", e
        )
        return pl.LazyFrame()
    except pl.exceptions.ComputeError as e:
        print("An error occurred while parsing the CSV data: %s", e)
        return pl.LazyFrame()


In [12]:
e_cleaning = load_gsheet_data(LOGISTICS_SHEET_ID, CONTAINER_CLEANING_SHEET)


e_pti = load_raw_sheet(LOGISTICS_SHEET_ID, CONTAINER_PTI_SHEET).select(
    pl.col('Timestamp').str.to_datetime(format="%d/%m/%Y %H:%M:%S").alias('timestamp'),
    pl.col("Date Plugin").str.to_date(format="%d/%m/%Y").dt.combine(pl.col("Time Plugin").str.to_time(format="%H:%M:%S")).alias("datetime_start"),
    pl.col("Container Number").alias("container_number"),
    pl.col("Size").alias("size"),
    pl.col("Set Point").cast(pl.Int16).alias("set_point"),
    pl.col("Unit Manufacturer").cast(pl.Categorical).alias("unit_manufacturer"),
    pl.col("Unplugged Date").str.to_datetime(format="%d/%m/%Y %H:%M",strict=False).alias("datetime_end"),
    pl.col("Sticker").str.replace("NA","N/A").cast(pl.Categorical).alias("sticker"),
    pl.col("Status").cast(pl.Categorical).alias("status"),
    pl.col("Shipping Line").str.replace("MAERSK","MAERSKLINE").cast(pl.Categorical).alias("invoice_to"),
    pl.col("Generator").cast(pl.Categorical).alias("plugged_on"),
)


In [22]:
e_cleaning.filter(pl.col("container_number").is_in(["MNBU3915695"])).collect()

Timestamp,date,container_number,shipping_line,cleaning_remarks,comments,Invoice To
datetime[μs],date,str,str,str,str,str
2025-06-03 13:31:55,2025-05-27,"""MNBU3915695""","""MAERSK""","""Clean""","""""","""MAERSKLINE"""
2025-06-18 14:20:42,2025-06-18,"""MNBU3915695""","""MAERSK""","""Unstuffed""","""""","""ATUNSA"""


In [8]:
search_id = "MNBU3927931"

e_pti.filter(pl.col("container_number").eq(pl.lit(search_id))).collect()

timestamp,datetime_start,container_number,size,set_point,unit_manufacturer,datetime_end,sticker,status,invoice_to,plugged_on
datetime[μs],datetime[μs],str,str,i16,cat,datetime[μs],cat,cat,cat,cat
2025-06-12 08:05:42,2025-06-11 16:45:00,"""MNBU3927931""","""40'""",-25,"""Starcool""",2025-06-12 09:07:00,"""PASS""","""PASSED""","""MAERSKLINE""","""K4"""


In [9]:
pti.collect()

datetime_start,container_number,set_point,invoice_to,datetime_end,hours,status,plugin_price,electricity_price,no_shifting,generator,shifting_price,total_price
datetime[μs],enum,enum,enum,datetime[μs],f64,enum,f64,f64,bool,str,f64,f64
2024-01-03 16:55:00,"""MMAU1082543""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MMAU1128460""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MNBU3645352""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MNBU3656948""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MNBU4208001""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-07-03 17:00:00,"""MNBU4545804""","""-25""","""MAERSKLINE""",2025-07-04 08:31:00,15.516667,"""PASSED""",25.0,70.0,true,"""AKSA""",35.0,130.0
2025-07-03 17:00:00,"""MNBU0152727""","""-25""","""MAERSKLINE""",2025-07-04 08:31:00,15.516667,"""PASSED""",25.0,70.0,true,"""AKSA""",35.0,130.0
2025-07-03 17:00:00,"""MNBU3990657""","""-25""","""MAERSKLINE""",2025-07-04 08:31:00,15.516667,"""PASSED""",25.0,70.0,true,"""AKSA""",35.0,130.0


In [12]:
e_pti.collect().write_clipboard()

C:\Users\gmounac\AppData\Local\Temp\ipykernel_12444\1646925375.py:1: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  e_pti.collect().write_clipboard()


In [28]:
current_month_non_pti.with_columns(
    container_number=pl.col("container_number").cast(pl.Utf8),
).join(e_pti.filter(pl.col("datetime_start").dt.month().eq(6)).collect(),on="container_number",how="left").write_clipboard()

C:\Users\gmounac\AppData\Local\Temp\ipykernel_23340\3670651272.py:3: CategoricalRemappingWarning: Local categoricals have different encodings, expensive re-encoding is done to perform this merge operation. Consider using a StringCache or an Enum type if the categories are known in advance
  ).join(e_pti.filter(pl.col("datetime_start").dt.month().eq(6)).collect(),on="container_number",how="left").write_clipboard()


### Search Container Number

MNBU3235635

In [11]:
container_num = "TRIU8660277"


e_cleaning.filter(pl.col("container_number").eq(pl.lit(container_num))).collect()

Timestamp,date,container_number,shipping_line,cleaning_remarks,comments,Invoice To
datetime[μs],date,str,str,str,str,str


### Same date duplication?

In [13]:
e_cleaning.filter(pl.col("Invoice To").ne(pl.lit("INVALID"))).with_columns(
    idx=pl.col("container_number") + pl.col("date").dt.to_string(format="%Y%m%d")
).filter(pl.col("idx").is_duplicated()).collect()

Timestamp,date,container_number,shipping_line,cleaning_remarks,comments,Invoice To,idx
datetime[μs],date,str,str,str,str,str,str


In [14]:
washing.filter(pl.col("invoice_to").ne(pl.lit("INVALID"))).with_columns(
    idx=pl.col("container_number") + pl.col("date").dt.to_string(format="%Y%m%d")
).filter(pl.col("idx").is_duplicated()).collect()

date,container_number,invoice_to,service_remarks,price,idx
date,enum,enum,str,i64,str
2025-05-09,"""SUDU6074313""","""MAERSKLINE""","""ADDED TO THE RECORD""",30,"""SUDU607431320250509"""
2025-05-09,"""SUDU8028528""","""MAERSKLINE""","""ADDED TO THE RECORD""",30,"""SUDU802852820250509"""
2025-05-09,"""SUDU6074313""","""MAERSKLINE""","""Clean""",30,"""SUDU607431320250509"""
2025-05-09,"""SUDU8028528""","""MAERSKLINE""","""Clean""",30,"""SUDU802852820250509"""
2025-05-20,"""MNBU3059080""","""MAERSKLINE""","""ADDED TO THE RECORD""",30,"""MNBU305908020250520"""
…,…,…,…,…,…
2025-06-25,"""MNBU3655221""","""MAERSKLINE""","""Clean""",30,"""MNBU365522120250625"""
2025-06-25,"""MNBU0353552""","""MAERSKLINE""","""Clean""",30,"""MNBU035355220250625"""
2025-06-25,"""MNBU3655221""","""MAERSKLINE""","""Clean""",30,"""MNBU365522120250625"""


### Washing Occuring in the same week

In [15]:
e_cleaning.filter(pl.col("Invoice To").ne(pl.lit("INVALID"))).with_columns(
    idx=pl.col("container_number") + pl.col("date").dt.to_string(format="%U")
).filter(pl.col("idx").is_duplicated()).filter(pl.col("date").dt.month().eq(6)).collect()

Timestamp,date,container_number,shipping_line,cleaning_remarks,comments,Invoice To,idx
datetime[μs],date,str,str,str,str,str,str


### Compare the Washing Service for a month

In [16]:
e_cleaning.filter(pl.col("Invoice To").eq(pl.lit("MAERSKLINE")).and_(pl.col("date").dt.month().eq(5))).join(
    washing.with_columns(pl.col("container_number").cast(pl.Utf8)).filter(
    pl.col("invoice_to").eq(pl.lit("MAERSKLINE")).and_(pl.col("date").dt.month().eq(6))
),on=["container_number", "date"], how="inner"
).collect()

Timestamp,date,container_number,shipping_line,cleaning_remarks,comments,Invoice To,invoice_to,service_remarks,price
datetime[μs],date,str,str,str,str,str,enum,str,i64


In [18]:
washing.filter(
    pl.col("invoice_to").eq(pl.lit("MAERSKLINE")).and_(pl.col("date").dt.month().eq(6))
).collect()

date,container_number,invoice_to,service_remarks,price
date,enum,enum,str,i64
2025-06-01,"""SUDU5204270""","""MAERSKLINE""","""Clean""",30
2025-06-01,"""SUDU6209189""","""MAERSKLINE""","""Clean""",30
2025-06-01,"""MNBU3514922""","""MAERSKLINE""","""Clean""",30
2025-06-01,"""MNBU4214827""","""MAERSKLINE""","""Clean""",30
2025-06-01,"""MNBU0303382""","""MAERSKLINE""","""Clean""",30
…,…,…,…,…
2025-06-26,"""MMAU1393334""","""MAERSKLINE""","""Clean""",30
2025-06-26,"""MNBU4139220""","""MAERSKLINE""","""Clean""",30
2025-06-26,"""MMAU1079302""","""MAERSKLINE""","""Clean""",30


In [13]:
washing.filter(pl.col("invoice_to").eq(pl.lit("MAERSKLINE")).and_(pl.col("date").dt.month().eq(5))).filter(pl.col("service_remarks").eq(pl.lit("ADDED TO THE RECORD"))).collect().write_clipboard()

In [14]:
container = "MNBU0650870"

In [13]:
e_pti.join(
    pti.filter(pl.col("datetime_start").dt.year().eq(2025)).with_columns(pl.col("container_number").cast(pl.Utf8)),on=["datetime_start","container_number"],how="full").filter(
        pl.col("datetime_start_right").is_null()).collect()

InvalidOperationError: conversion from `str` to `datetime[μs]` failed in column 'Timestamp' for 1 out of 125 values: [""]

You might want to try:
- setting `strict=False` to set values that cannot be converted to `null`
- using `str.strptime`, `str.to_date`, or `str.to_datetime` and providing a format string

In [14]:
pti.collect()

datetime_start,container_number,set_point,invoice_to,datetime_end,hours,status,plugin_price,electricity_price,no_shifting,generator,shifting_price,total_price
datetime[μs],enum,enum,enum,datetime[μs],f64,enum,f64,f64,bool,str,f64,f64
2024-01-03 16:55:00,"""MMAU1082543""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MMAU1128460""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MNBU3645352""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MNBU3656948""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
2024-01-03 16:55:00,"""MNBU4208001""","""-25""","""MAERSKLINE""",2024-01-04 10:00:00,17.083333,"""PASSED""",25.0,70.0,true,"""K3""",35.0,130.0
…,…,…,…,…,…,…,…,…,…,…,…,…
2025-06-01 15:45:00,"""MMAU1158660""","""-25""","""MAERSKLINE""",2025-06-02 10:07:00,18.366667,"""FAILED""",25.0,70.0,true,"""AKSA""",35.0,130.0
2025-06-01 15:45:00,"""MNBU3761579""","""-25""","""MAERSKLINE""",2025-06-02 10:07:00,18.366667,"""FAILED""",25.0,70.0,true,"""AKSA""",35.0,130.0
2025-06-01 15:45:00,"""MNBU3438360""","""-25""","""MAERSKLINE""",2025-06-02 10:07:00,18.366667,"""PASSED""",25.0,70.0,true,"""AKSA""",35.0,130.0


In [17]:
e_cleaning.filter(pl.col("container_number").eq(pl.lit(container))).collect()

Timestamp,date,container_number,shipping_line,cleaning_remarks,comments,Invoice To
datetime[μs],date,str,str,str,str,str


In [15]:
e_pti.filter(pl.col("container_number").eq(pl.lit(container))).collect()

NameError: name 'container' is not defined

In [18]:
e_pti.collect()

InvalidOperationError: conversion from `str` to `datetime[μs]` failed in column 'Timestamp' for 1 out of 125 values: [""]

You might want to try:
- setting `strict=False` to set values that cannot be converted to `null`
- using `str.strptime`, `str.to_date`, or `str.to_datetime` and providing a format string